In [60]:
#Import Packages
import functools
import random
from copy import copy
import pettingzoo
import numpy as np
from gymnasium.spaces import Discrete, MultiDiscrete
from pettingzoo import ParallelEnv
import matplotlib.pyplot as plt
from pettingzoo.test import parallel_api_test
from gymnasium import spaces

In [63]:
#Environment Definition
#Need parallel environment for this as multiple agents can act at once
from pettingzoo import ParallelEnv
class CustomEnvironment(ParallelEnv):
    metadata = {
        "name": "multi_satellite_downlink_v0",
    }
    def __init__(self):
        """The init method takes in environment arguments.

        Note: as of v1.18.1, the action_spaces and observation_spaces attributes are deprecated.
        Spaces should be defined in the action_space() and observation_space() methods.
        If these methods are not overridden, spaces will be inferred from self.observation_spaces/action_spaces, raising a warning.

        These attributes should not be changed after initialization.
        """
        self.master_contact_plan = None
        self.satellite_plan = None
        self.satellite_data=[None,None,None]
        self.timestep = None
        self.possible_agents = ["satellite1","satellite2","satellite3"]
        self.delivery_ratio=None
        self.energy_efficiency=None
        self.energy_expenditure=None
        self.delivered_packets=None
        self.initial_satellite_data=None
        self.original_matrix=None
        self.obs_satellite_data=[None,None,None;None,None,None;None,None,None]
        obs_space=spaces.Dict({
            "central_observation_matrix": spaces.Box(low=0.0, high=1.0, shape=(30, 5), dtype=np.float32),
            "remaining_satellite_data": spaces.Box(low=-1.0, high=1.0, shape=(3,), dtype=np.float32),
            "current_timestep": spaces.Box(low=0.0, high=32, shape=(1,), dtype=np.float32),
            "action_mask": spaces.MultiBinary(2),
        })
        act_space=spaces.Discrete(1)
        self.observation_spaces = {agent: obs_space for agent in self.possible_agents}
        self.action_spaces = {agent: act_space for agent in self.possible_agents}

    def reset(self, seed=None, options=None):
        """Reset set the environment to a starting point.

        It needs to initialize the following attributes:
        - agents
        - timestamp
        - Initial Data Volume
        - Master Contact Plan
        - Number of Delivered Packets
        - Energy Expenditure
        - Delivery Ratio
        - Energy Efficiency
        - observation
        - infos

        And must set up the environment so that render(), step(), and observe() can be called without issues.
        """
        self.agents = copy(self.possible_agents)
        self.timestep = 0
        #Define the initial data volume at each satellite
        for i in range(3):
            self.satellite_data[i]=random.randint(5,100)
        print(self.satellite_data)
        self.initial_satellite_data=0
        for i in range(len(self.satellite_data)):
            self.initial_satellite_data=self.initial_satellite_data+self.satellite_data[i]
        #Define the master contact plan
        #Contains 30 contacts (total), weather values for each contact and which contacts are shared
            
        #Code below was written with a template created with ChatGPT
        nodes = ['Satellite1', 'Satellite2', 'Satellite3']
        max_rows_per_node = 10
        node_usage = {node: 0 for node in nodes}
        
        matrix = []
        
        row_index = 1
        while any(count < max_rows_per_node for count in node_usage.values()):
            # Get nodes that still have availability
            available_nodes = [node for node in nodes if node_usage[node] < max_rows_per_node]
        
            # Determine how many nodes to assign in this row
            max_possible = len(available_nodes)
            num_nodes_in_row = random.randint(1, max_possible)
        
            # Select nodes for the row
            selected_nodes = random.sample(available_nodes, num_nodes_in_row)
        
            # Update usage count
            for node in selected_nodes:
                node_usage[node] += 1
        
            # Create and store the row
            row = [random.random(), 10, selected_nodes]
            matrix.append(row)
        
            row_index += 1
        # Output the matrix
        #for row in matrix:
            #print(row)
        
        # Show final node usage
        #print("\nFinal node usage:")
        #for node, count in node_usage.items():
            #print(f"{node}: {count}")
        
        #print(f"\nTotal rows generated: {len(matrix)}")

        #Next we need either divide up the master contact plan during observations or 
        #Pad the complete matrix up to 30
        
        if len(matrix)<30:
            rows, cols = 30, 3
            new_matrix = [[None for _ in range(cols)] for _ in range(rows)]
            for i in range(30):
                if i>(len(matrix)-1):
                    new_matrix[i]=[-1,-1,-1]
                else:
                    #print(i)
                    #print(new_matrix)
                    new_matrix[i]=matrix[i]
        
        
        else:
            new_matrix=matrix
            #print(new_matrix)

        self.original_matrix=new_matrix
        #Next we must convert the new matrix into a format that can be loaded into the observation space effectively
        rows, cols = 30, 5
        obs_matrix = [[None for _ in range(cols)] for _ in range(rows)]
        for i in range(30):
            for j in range(3):
                if j<2:
                    obs_matrix[i][j]=new_matrix[i][j]
                elif j==2:
                    current_element=new_matrix[i][j]
                    #print(current_element)
                    encoded_row=[0,0,0]
                    if current_element!=-1:
                        for k in range(len(current_element)-1):
                            if current_element[k]=="satellite1":
                                encoded_row[0]=1
                            elif current_element[k]=="satellite2":
                                encoded_row[1]=1
                            elif current_element[k]=="satellite3":
                                encoded_row[2]=1
                    else:
                        encoded_row=[0,0,0]
                    for k in range(2,5):
                        #print("Encoded Row")
                        #print(encoded_row)
                        obs_matrix[i][k]=encoded_row[k-2]
        #print("Obs matrix")
        #print(obs_matrix)
        #Next we must define the action mask 
        action_mask=[[0,0],[0,0],[0,0]]
        if obs_matrix[1][2]==1:
            action_mask[0]:[1,1]
        if obs_matrix[1][3]==1:
            action_mask[1]:[1,1]
        if obs_matrix[1][4]==1:
            action_mask[2]:[1,1]
        #Next need to configure action mask as dictionary
        obs_action_mask = {a: action_mask[i] for i, a in enumerate(self.agents)}
        #for a in self.agents:
        #    obs_action_mask=dict[a:None]
        #print(obs_action_mask)
        #i=0
        #for a in self.agents:
        #    obs_action_mask[a]=action_mask[i]
        #    i=i+1
        #print(obs_action_mask)


        
                #randomly allocate weather conditions and if the contact is shared
        self.timestep=1
        self.master_contact_plan=obs_matrix        
        #print(self.master_contact_plan)
        #Set delivery ratio for each satellite
        self.delivery_ratio=[None,None,None]
        self.delivered_packets=[0,0,0]
        #Set energy expenditure for each satellite
        self.energy_efficiency=[None,None,None]
        self.energy_expenditure=[0,0,0]
        
        
        #Implement observations
        #Each satellite has access to the following observations
        #The total contact matrix
        #The amount of data remaining in the each satellite (at the last point we contacted the ground)
        #The current timestep
        #The action mask
        

        
        observations = {
            a: {
                "central_observation_matrix":obs_matrix,
                "remaining_satellite_data":self.obs_satellite_data[a],
                "current_timestep":self.timestep,
                "action_mask":obs_action_mask[a]
            }
            for a in self.agents
        }

        # Get dummy infos. Necessary for proper parallel_to_aec conversion
        infos = {a: {} for a in self.agents}
        print("self.agents:", self.agents)
        print("Returning observations for:", list(observations.keys()))
        print("Expected agents:", self.agents)
        missing = [agent for agent in self.agents if agent not in observations]
        if missing:
            print("❌ Missing observations for agents:", missing)
        return observations, infos

    def step(self, actions):
        """Takes in an action for the current agent (specified by agent_selection).

        Needs to update:
        - prisoner x and y coordinates
        - guard x and y coordinates
        - terminations
        - truncations
        - rewards
        - timestamp
        - infos

        And any internal state used by observe() or render()
        """
        # Execute actions
        #First we must get the actions for each satellite
        satellite_actions = [actions[a] for a in ["satellite1", "satellite2", "satellite3"]]
        #print("Satellite Actions")
        #print(satellite_actions)
        #Next we update the delivery and energy expenditure for all satellites
        
        
        #This logic was partially aided by ChatGPT (used to it to get basic structure of checking if satellites are competing for ground stations, however reward and majority of logic is my own)
        delivered_packets=0
        excess_energy=0
        total_ones = satellite_actions.count(1)
        #Next we check if there are any collisions between the agents
        collision_matrix=self.checkActionConflict(self.timestep,satellite_actions)


        rewards={}
        i=0
        #Need to fix this
        for a in self.agents:
            current_value = actions[a]
            other_ones_exist = (total_ones - (1 if current_value == 1 else 0)) > 0
            
            if collision_matrix[i]==1:
                #We have a conflict, no packets are sent and the satellite receives a large penalty
                reward[a]=-10
            else:
                if current_value==0:
                    rewards[a]=0
                else:
                    delivered_packets,excess_energy=updateDeliveryandEnergy(self,agent,row[1],row[2])
                    if delivered_packets >0:
                        #Positive Reward
                        rewards[a]=delivered_packets/(excess_energy+1)
                        #Next need to update the observed satellite data values for each satellite
                        interim_satellite_data_observations=satellite_obs_update(actions,delivered_packets)
                    else:
                        rewards[a]=-1
            i=i+1
       
        
        terminations={a:False for a in self.agents}
        truncations={a: False for a in self.agents}
        #Next we must update the parameters changed in step function
        self.updateTimestep()
        #Next we must update the number of delivered packets and energy expenditur
        #Next we update the contact plan representation
        #Outline conditions for terminations
        if self.timestep>=30 :
        #All satellites have 
            truncations={a: True for a in self.agents}
            new_matrix=self.original_matrix
            rows, cols = 30, 5
            obs_matrix = [[None for _ in range(cols)] for _ in range(rows)]
            for i in range(30):
                for j in range(3):
                    if j<2:
                        obs_matrix[i][j]=new_matrix[i][j]
                    elif j==2:
                        current_element=new_matrix[i][j]
                        #print(current_element)
                        encoded_row=[0,0,0]
                        if current_element!=-1:
                            for k in range(len(current_element)-1):
                                if current_element[k]=="satellite1":
                                    encoded_row[0]=1
                                elif current_element[k]=="satellite2":
                                    encoded_row[1]=1
                                elif current_element[k]=="satellite3":
                                    encoded_row[2]=1
                        else:
                            encoded_row=[0,0,0]
                        for k in range(2,5):
                            #print("Encoded Row")
                            #print(encoded_row)
                            obs_matrix[i][k]=encoded_row[k-2]
            
            #print("Obs matrix")
            #print(obs_matrix)
            #Next we must define the action mask 
            action_mask=[[0,0],[0,0],[0,0]]
            if obs_matrix[29][2]==1:
                action_mask[0]:[0,0]
            if obs_matrix[29][3]==1:
                action_mask[1]:[0,0]
            if obs_matrix[29][4]==1:
                action_mask[2]:[0,0]
            obs_action_mask = {a: action_mask[i] for i, a in enumerate(self.agents)}
        else:
            new_matrix=self.original_matrix
            rows, cols = 30, 5
            obs_matrix = [[None for _ in range(cols)] for _ in range(rows)]
            for i in range(30):
                for j in range(3):
                    if j<2:
                        obs_matrix[i][j]=new_matrix[i][j]
                    elif j==2:
                        current_element=new_matrix[i][j]
                        #print(current_element)
                        encoded_row=[0,0,0]
                        if current_element!=-1:
                            for k in range(len(current_element)-1):
                                if current_element[k]=="satellite1":
                                    encoded_row[0]=1
                                elif current_element[k]=="satellite2":
                                    encoded_row[1]=1
                                elif current_element[k]=="satellite3":
                                    encoded_row[2]=1
                        else:
                            encoded_row=[0,0,0]
                        for k in range(2,5):
                            #print("Encoded Row")
                            #print(encoded_row)
                            obs_matrix[i][k]=encoded_row[k-2]
            
            #print("Obs matrix")
            #print(obs_matrix)
            #Next we must define the action mask 
            action_mask=[[0,0],[0,0],[0,0]]
            if obs_matrix[self.timestep][2]==1:
                action_mask[0]:[1,1]
            if obs_matrix[self.timestep][3]==1:
                action_mask[1]:[1,1]
            if obs_matrix[self.timestep][4]==1:
                action_mask[2]:[1,1]
            obs_action_mask = {a: action_mask[i] for i, a in enumerate(self.agents)}
        if delivered_packets==self.initial_satellite_data:
            terminations={a: True for a in self.agents}
        

        infos = {a: {} for a in self.agents}
        
        if any(terminations.values()) or all(truncations.values()):
            self.agents = []
        observations = {
            a: {
                "central_observation_matrix":obs_matrix,
                "delivered_packets":self.delivered_packets,
                "energy_expenditure":self.energy_expenditure,
                "remaining_satellite_data":self.satellite_data,
                "current_timestep":self.timestep,
                "action_mask":obs_action_mask[a]
                #self.prisoner_x + 7 * self.prisoner_y,
                #self.guard_x + 7 * self.guard_y,
                #self.escape_x + 7 * self.escape_y,

                #Need to implement action mask
                
            }
            for a in self.agents
        }
        print("self.agents:", self.agents)
        print("Returning observations for:", list(observations.keys()))
        print("Expected agents:", self.agents)
        missing = [agent for agent in self.agents if agent not in observations]
        if missing:
            print("❌ Missing observations for agents:", missing)
        return observations, rewards, terminations, truncations, infos
    def getRow(self,timestep):
        return self.master_contact_plan[timestep]
    #Function to simply update timestep
    #Where the timestep is the contact in the contact plan
    def updateTimestep(self):
        self.timestep=self.timestep+1
    #Get current timestep
    def getTimestep(self):
        return self.timestep
    #In this function we obtain the number of delivered packets and excess energy expenditure 
    #If we select the given contact
    def updateDeliveryandEnergy(self,agent,weather,length):
        delivered_packets=0
        excess_energy_expended=0
        for i in range(length)-1:
            random_sample=random.random()
            if random_sample> weather:
                delivered_packets=delivered_packets+1
            else:
                excess_energy_expended=excess_energy_expended+1
    
        return delivered_packets,excess_energy_expended
    #Check if any satellites have conflicts in terms of connections
    def checkActionConflict(self,timestep,actions):
        master_observation_matrix=self.master_contact_plan
        print(timestep)
        LoS_matrix=master_observation_matrix[timestep][2:4]
        #Next we must consider the actions
        conflict_matrix=[0,0,0]
        
        for j in range(2):
            if actions[j]*LoS_matrix[j]==1:
                conflict_matrix[j]=1
        penalty_matrix=[0,0,0]
        if sum(conflict_matrix)>1:
            penalty_matrix=conflict_matrix
        return penalty_matrix
    #Update the observations of the satellites if one makes contact with the ground
    def satellite_obs_update(actions,delivered_packets):
        #We take the actions and check that we established contact with the ground station
        #We update for the successful satellite the satellite data values that are known by the network
        #And we update the array that holds it for subsequent environment steps
        



        
def render(self):
        """Renders the environment."""
        #We will output a bar graph with the current delivery and energy expenditure
        

# Scalar values (e.g., from a list)
        values = [3, 7, 2, 5, 9]

# Optional: Labels for each bar (x-axis)
        labels = ["satellite1 Delivery Ratio","satellite2 Delivery Ratio","satellite3 Delivery Ratio","satellite1 Energy Efficiency","satellite2 Energy Efficiency","satellite3 Energy Efficiency"]

# Create the bar chart
        plt.bar(labels, values)

# Add labels and title
        plt.xlabel('KPI results')
        plt.title('System Performance')

# Show the chart
plt.show()
    # Observation space should be defined here.
    # lru_cache allows observation and action spaces to be memoized, reducing clock cycles required to get each agent's space.
    # If your spaces change over time, remove this line (disable caching).
#@functools.lru_cache(maxsize=None)
def observation_space(self, agent):
    # gymnasium spaces are defined and documented here: https://gymnasium.farama.org/api/spaces/
   return spaces.Dict({
            "central_observation_matrix": spaces.Box(low=0.0, high=1.0, shape=(30, 5), dtype=np.float32),
            "delivered_packets": spaces.Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "energy_expenditure": spaces.Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "remaining_satellite_data": spaces.Box(low=-1.0, high=1.0, shape=(3,), dtype=np.float32),
            "current_timestep": spaces.Box(low=0.0, high=32, shape=(1,), dtype=np.float32),
            "action_mask": spaces.MultiBinary(2),
        })
    

# Action space should be defined here.
# If your spaces change over time, remove this line (disable caching).
#@functools.lru_cache(maxsize=None)
def action_space(self, agent):
    return Discrete(1)

#Function to retrieve a single row from the master contact plan



In [64]:
env = CustomEnvironment()
parallel_api_test(env, num_cycles=1_000_000)

[79, 36, 12]
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
[57, 85, 16]
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
1
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
2
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
3
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
4
self.agents: ['satellite1', 'sa

In [ ]:
#Next we must setup the agent training and definition
"""Uses Stable-Baselines3 to train agents to play the Waterworld environment using SuperSuit vector envs.

For more information, see https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html

Author: Elliot (https://github.com/elliottower)
"""
from __future__ import annotations

import glob
import os
import time

import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy



def train_PPO(
    env_func, steps: int = 10_000, seed: int | None = 0, **env_kwargs
):
    # Train a single model to play as each agent in a cooperative Parallel environment
    env = env_fn.parallel_env(**env_kwargs)

    env.reset(seed=seed)

    print(f"Starting training on {str(env.metadata['name'])}.")

    env =
    env =

    # Note: Waterworld's observation space is discrete (242,) so we use an MLP policy rather than CNN
    model = PPO(
        MlpPolicy,
        env,
        verbose=3,
        learning_rate=1e-3,
        batch_size=256,
    )

    model.learn(total_timesteps=steps)

    model.save(f"{env.unwrapped.metadata.get('name')}_{time.strftime('%Y%m%d-%H%M%S')}")

    print("Model has been saved.")

    print(f"Finished training on {str(env.unwrapped.metadata['name'])}.")

    env.close()


def eval(env_fn, num_games: int = 100, render_mode: str | None = None, **env_kwargs):
    # Evaluate a trained agent vs a random agent
    env = env_fn.env(render_mode=render_mode, **env_kwargs)

    print(
        f"\nStarting evaluation on {str(env.metadata['name'])} (num_games={num_games}, render_mode={render_mode})"
    )

    try:
        latest_policy = max(
            glob.glob(f"{env.metadata['name']}*.zip"), key=os.path.getctime
        )
    except ValueError:
        print("Policy not found.")
        exit(0)

    model = PPO.load(latest_policy)

    rewards = {agent: 0 for agent in env.possible_agents}

    # Note: We train using the Parallel API but evaluate using the AEC API
    # SB3 models are designed for single-agent settings, we get around this by using the same model for every agent
    for i in range(num_games):
        env.reset(seed=i)

        for agent in env.agent_iter():
            obs, reward, termination, truncation, info = env.last()

            for a in env.agents:
                rewards[a] += env.rewards[a]
            if termination or truncation:
                break
            else:
                act = model.predict(obs, deterministic=True)[0]

            env.step(act)
    env.close()

    avg_reward = sum(rewards.values()) / len(rewards.values())
    print("Rewards: ", rewards)
    print(f"Avg reward: {avg_reward}")
    return avg_reward


if __name__ == "__main__":
    env_fn = waterworld_v4
    env_kwargs = {}

    # Train a model (takes ~3 minutes on GPU)
    #train_butterfly_supersuit(env_fn, steps=196_608, seed=0, **env_kwargs)
    train_PPO(env_func,steps=,seed=0, **env_kwargs)
    # Evaluate 10 games (average reward should be positive but can vary significantly)
    #eval(env_fn, num_games=10, render_mode=None, **env_kwargs)

    # Watch 2 games
    #eval(env_fn, num_games=2, render_mode="human", **env_kwargs)

In [ ]:
#Next we evaluate the agent performance
#Start with independent PPO
#Then explore more advanced strategies
